In [ ]:
import json
import re
import os

def parse_triple(triple):
    pattern = re.compile(r"\[entity1, ([^\]]+)\]: (.+?), \[relation, ([^\]]+)\]: (.+?), \[entity2, ([^\]]+)\]: (.+)")
    match = pattern.search(triple["triples"])
    if match:
        entity1_id, entity1, relation_id, relation, entity2_id, entity2 = match.groups()
        return (entity1_id, entity1.strip()), (relation_id, relation.strip()), (entity2_id, entity2.strip())
    else:
        raise ValueError("Triple format is incorrect")

def count_unique_entities_in_folder(folder_path):
    unique_entities = set()
    for filename in os.listdir(folder_path):
        if filename.endswith('.jsonl'):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r', encoding='utf-8') as file:
                for line in file:
                    triple_data = json.loads(line)
                    try:
                        entity1, relation, entity2 = parse_triple(triple_data)
                        unique_entities.update([entity1[1], entity2[1]])  # Only store the entity names
                    except ValueError as e:
                        print(f"Error parsing file {file_path}: {e}")
    
    return len(unique_entities)

folder_path = 'entity_rels_reformed'
print(f"Total number of unique entities across all files in the folder: {count_unique_entities_in_folder(folder_path)}")


In [ ]:
import os
from tqdm import tqdm
import re

input_folder1 = 'generated_sentences'
input_folder2 = 'entity_rels_reformed'
output_folder = 'text2kg'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

files1 = os.listdir(input_folder1)
files2 = os.listdir(input_folder2)

files1_dict = {re.search(r'\d+', f).group(0): f for f in files1 if f.endswith('.jsonl')}
files2_dict = {re.search(r'\d+', f).group(0): f for f in files2 if f.endswith('.jsonl')}

total_files = len(files2_dict)
progress_bar = tqdm(total=total_files, desc="Merging files")

for key, file2 in files2_dict.items():
    file1 = files1_dict.get(key)
    if file1:
        file_path1 = os.path.join(input_folder1, file1)
        file_path2 = os.path.join(input_folder2, file2)
        output_file_path = os.path.join(output_folder, f"merged_{key}.txt")

        with open(file_path1, 'r', encoding='utf-8') as file1, \
             open(file_path2, 'r', encoding='utf-8') as file2, \
             open(output_file_path, 'w', encoding='utf-8') as output_file:

            lines1 = file1.readlines()
            lines2 = file2.readlines()

            for line1, line2 in zip(lines1, lines2):
               
                modified_line2 = '@' + line2.strip()
               
                merged_line = line1.strip() + ' ' + modified_line2
                
                cleaned_line = merged_line.replace('"', '').replace('{', '').replace('}', '') + '\n'
               
                output_file.write(cleaned_line)

      
        progress_bar.update(1)


progress_bar.close()

print("finish")


In [ ]:
import os
import re
import unicodedata
from tqdm import tqdm

output_folder = 'text2kg'

txt_files = [f for f in os.listdir(output_folder) if f.endswith('.txt')]

progress_bar = tqdm(total=len(txt_files), desc="Processing files")

def replace_unicode_and_symbols(text):
    
    normalized_text = unicodedata.normalize('NFKD', text)
    
    ascii_text = normalized_text.encode('ascii', 'ignore').decode('ascii')
 
    ascii_text = ascii_text.replace('# @', '#@')
    return ascii_text

for txt_file in txt_files:
    file_path = os.path.join(output_folder, txt_file)
    
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    with open(file_path, 'w', encoding='utf-8') as f:
        for line in lines:
            cleaned_line = replace_unicode_and_symbols(line)
            f.write(cleaned_line)
    
    progress_bar.update(1)

progress_bar.close()

print("finish")


In [ ]:
import os
from tqdm import tqdm

output_folder = 'text2kg'
merged_file_path = os.path.join(output_folder, 'merged_output.txt')

txt_files = [f for f in os.listdir(output_folder) if f.endswith('.txt')]

progress_bar = tqdm(total=len(txt_files), desc="Processing files")

with open(merged_file_path, 'w', encoding='utf-8') as merged_file:

    for txt_file in txt_files:
        file_path = os.path.join(output_folder, txt_file)
        
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        for line in lines:
            merged_file.write(line.strip() + '@\n')
        
        progress_bar.update(1)

progress_bar.close()

print("finish")


In [ ]:
def print_first_5_lines(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for i in range(5):
                line = file.readline()
                if line: 
                    print(line.strip())
                else: 
                    break
    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except Exception as e:
        print(f"An error occurred: {e}")

huge_file_path = 'text2kg/merged_output.txt'

print_first_5_lines(huge_file_path)
